# H1: Local Cost Functions to Mitigate Barren Plateaus

**Hypothesis**: Replacing global expectation-value loss with a weighted sum of local observables reduces gradient variance and speeds training on small QML classification tasks.

In [ ]:
import pennylane as qml
import numpy as np
import json
import os
from datetime import datetime

np.random.seed(42)
qml.numpy.random.seed(42)

N_QUBITS = 4
N_LAYERS = 2
N_SAMPLES = 80
N_EPOCHS = 50
LR = 0.05
BATCH_SIZE = 20

dev = qml.device('lightning.qubit', wires=N_QUBITS, shots=None)

def global_ansatz(x, params):
    for i in range(N_QUBITS):
        qml.RX(x[i % len(x)], wires=i)
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(params[l * 2 * N_QUBITS + i], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(N_QUBITS):
            qml.RZ(params[l * 2 * N_QUBITS + N_QUBITS + i], wires=i)

@qml.qnode(dev)
def global_cost_circuit(x, params):
    global_ansatz(x, params)
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1) @ qml.PauliZ(2) @ qml.PauliZ(3))

@qml.qnode(dev)
def local_cost_circuit(x, params):
    global_ansatz(x, params)
    obs = [qml.PauliZ(i) for i in range(N_QUBITS)]
    return qml.expval(qml.sum(*obs) / N_QUBITS)

In [ ]:
def generate_dataset(n_samples, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, size=(n_samples, 2))
    y = np.array([1 if x[0] * x[1] > 0 else -1 for x in X])
    return X, y

def train(circuit, X, y, label):
    n_params = N_LAYERS * 2 * N_QUBITS
    params = np.random.default_rng(42).uniform(-0.1, 0.1, size=n_params)
    opt = qml.GradientDescentOptimizer(stepsize=LR)
    losses = []
    for epoch in range(N_EPOCHS):
        idx = np.random.default_rng(epoch).permutation(len(X))
        epoch_loss = 0.0
        for start in range(0, len(X), BATCH_SIZE):
            batch_idx = idx[start:start + BATCH_SIZE]
            batch_X, batch_y = X[batch_idx], y[batch_idx]
            def loss_fn(p):
                preds = np.array([circuit(x, p) for x in batch_X])
                return np.mean((preds - batch_y) ** 2)
            params, cost = opt.step_and_cost(loss_fn, params)
            epoch_loss += cost
        avg_loss = epoch_loss / max(1, (len(X) // BATCH_SIZE))
        losses.append(float(avg_loss))
        if epoch % 10 == 0:
            preds = np.array([circuit(x, params) for x in X])
            acc = np.mean((np.sign(preds) == y).astype(float))
            print(f"[{label}] Epoch {epoch:3d}  loss={avg_loss:.4f}  acc={acc:.3f}")
    final_preds = np.array([circuit(x, params) for x in X])
    final_acc = float(np.mean((np.sign(final_preds) == y).astype(float)))
    return {'params': params.tolist(), 'losses': losses, 'final_acc': final_acc, 'label': label}

X, y = generate_dataset(N_SAMPLES)
os.makedirs('results/H1', exist_ok=True)

print("=== Training global cost circuit ===")
global_result = train(global_cost_circuit, X, y, 'global')

print("\n=== Training local cost circuit ===")
local_result = train(local_cost_circuit, X, y, 'local')

results = {'global': global_result, 'local': local_result}
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'results/H1/run_{ts}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to results/H1/run_{ts}.json")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(global_result['losses'], label='Global cost', marker='o', ms=3)
plt.plot(local_result['losses'], label='Local cost', marker='s', ms=3)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('H1: Global vs Local Cost Training Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
os.makedirs('results/H1', exist_ok=True)
plt.savefig('results/H1/plot.png', dpi=120)
plt.show()
print(f"Final accuracy global={global_result['final_acc']:.3f}  local={local_result['final_acc']:.3f}")